# Intro to Spark

**Disclaimer:** TP plus compliqué !

**Spark** est un système de calcul hautement parallélisé :
- au niveau du stockage : la data est fragmentée, dupliquée, répartie sur un nombre quelconque de disques/servers
- au niveau du calcul : les calculs s'exécutent en parallèle sur plusieurs machines, chacune sur sa portion de data

Spark en tant que tel est un framework qui a des implémentation dans plusieurs langages :
- Python via PySpark (ce qe nous utiliserons)
- Scala
- R
- Java

## Quelques informations en vrac sur Spark
### Hardware
- Spark peut s'installer sur un grand nombre de machines situées dans un même réseau. Elles pourront ensuite se reconnaître et collaborer. 1 unité de calcul = 1 noeud.
- Spark fonctionne sur le mode master/worker : un noeud est désigné `master` et jouera le rôle de chef d'orchestre pour que les autres noeuds `workers` exécutent les tâches dans le bon ordre
- Les noeuds Spark communiquent énormément entre eux pour s'échanger des informations et surtout des données

### Software
- Un code Spark / PySpark doit utiliser les primitives Spark pour que tout s'exécute selon la logique Spark
- Le code pyspark est transmis au noeud `master` qui le lit et prépare l'orchestration des calculs selon les `workers` qu'il a à disposition. Seuls les `workers` manipuleront la donnée (sauf exception)
- Spark est *lazy* : `master` ne lance réellement aucun calcul tant qu'il n'a pas lu d'opération impliquant l'affichage ou l'écriture des résultats
- Corrolaire du *lazy* : sans précaution, Spark peut répéter plusieurs fois les mêmes calculs ... Exemple avec 2 chaînes de transformation data `A -> B -> C -> D` suivi de `A -> B -> E`. Les étapes intermédiaires `A -> B` sont identiques mais pour calculer `D` et `E`, Spark risque de les exécuter 2 fois. Apprendre à manipuler les méthodes [cache](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.cache.html?highlight=cache) !

### Framework
- Spark se base sur des `DataFrame` très proches en terme d'utilisation des `pandas.DataFrame` donc pas de panique :)
- Spark gère tout ce que sait gérer SQL mais part de fichiers plats (ici CSV) : join, select, etc ...
- PySpark permet de gérer tout ces concepts avec la facilité d'accès du Python
- PySpark est TRÈS typé et a besoin de connaître les types de chaque colonne manipulées
- Spark est TRÈS flexible en terme de configuration et d'exécution et cela peut sembler déroutant pour des exemples simples

### Organisation du calcul
En informatique, à chaque architecture ses optimisations :
- en local sur 1 CPU, un calcul possède peu d'optimisation : simple et linéaire, possiblement async
- en local sur multi CPU, un calcul doit être prévu pour paralléliser les exécutions sur plusieurs coeurs physiques
- en local sur mono/multi GPU, un calcul doit être hautement parallélisable, découpable en tranche de data qui tiennent en GPU-RAM
- en multi machine multi GPU, idem que plus haut avec bande passante importante entre machine pour échange d'information

... Spark gère le multi machine, multi CPU, multi RAM, multi disque : calculs hautement parallélisés grâce à la magie de Spark, data échangées au mieux entre machine (possiblement avec l'aide humaine).
__Spark est toujours prêt à gérer un contexte d'exécution très complexe__ => il faut s'attendre à beaucoup d'overhead sur des cas simples

**Les opérations s'exécutent sur des workers séparés, en parallèle**, il faut donc parfois faire "un peu attention" à la façon dont on demande à Spark de *partitionner* sa data.

## À retenir

1. Spark et son implémentation PySpark sont très puissant car gèrent un parallélisme quasi infini et réglable à 100%
2. PySpark a un coût d'entrée pour se couler dans le moule Spark mais permet de réaliser des opérations très complexes avec la simplicité du Python
3. Votre notebook n'exécutera n'a pas accès à la data manipulée et n'effectuera aucun calculs ; il les transmettra au Spark Master qui les répartira entre ses workers qui ont accès à la data

## Exemple de code Spark

In [1]:
import os
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
MY_ID = "François"

In [3]:
# Initialize PySpark session
assert MY_ID is not None, "provide your id first!"
spark = SparkSession.builder \
    .appName(MY_ID) \
    .master(os.getenv('SPARK_MASTER')) \
    .config('spark.executor.cores', '2') \
    .config('spark.executor.memory', '500m') \
    .config('spark.executor.instances', '2') \
    .config('spark.default.parallelism', '2') \
    .config('spark.sql.shuffle.partitions', '2') \
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262') \
    .config('spark.hadoop.fs.s3a.endpoint', os.getenv('S3_ENDPOINT', 'http://minio:9000')) \
    .config('spark.hadoop.fs.s3a.access.key', os.getenv('S3_ACCESS_KEY', 'minioadmin')) \
    .config('spark.hadoop.fs.s3a.secret.key', os.getenv('S3_SECRET_KEY', 'minioadmin')) \
    .config('spark.hadoop.fs.s3a.path.style.access', 'true') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false') \
    .getOrCreate()

# /!\ Tout se fera à partir de cet object magique `spark`

OSError: [Errno 28] No space left on device: '/tmp/tmpyj5j55ik'

In [ ]:
%%time
data = [("Alice", 1), ("Bob", 2), ("Catherine", 3)]
df = spark.createDataFrame(data, ["Name", "Value"])

# Show the dataframe
df.show()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, FloatType, ArrayType, StructField, StructType, DoubleType

In [ ]:
df_beers = spark.read.parquet("s3a://beer-dataset/parquet/beers")
df_beers.show()

In [ ]:
%%time
# Define a specific funtion to map beer names
@F.udf(returnType=StringType())
def revert_cap_name(name: str):
    return name[::-1].upper()


df_beers.repartition(12).withColumn("reverse_capitalized", revert_cap_name(F.col("name"))).select("name", "reverse_capitalized").show()

## Observations
Que remarque-t-on tout de suite ?

# Uses cases

# UC-1 : description data

- Q1: Combien y a-t-il de bières dans la DB ?
- Q2: Top10 brasseries les plus représentées avec le nombre de bière par brasserie ?
- Q3: Top10 des bières les plus fortes (ABV) en France ?
- Q4: Par pays, nombre de brasseries qui proposent des bières de type `Porter` et ABV moyen de celles-ci ?
- Q5: Mediane du nombre de bière par pays ?

# UC-2 : préparer un dataset de ranking 
Tout moteur de recherche/search-engine - **SE** - nécessite de la configuration ... beaucoup de configuration. Une des configuration très orientée "data" est le calcul que l'index doit opérer pour scorer chaque réponse possible face à une requête. L'apprentissage statistique de ce score s'appelle *Learning to Rank*  - **LTR** - et nécessite des connaissances poussées en machine learning. 

Cette tâche LTR se base sur les *feedbacks implicites* des utilisateurs face au moteur de recherche. Commençons par un exemple. Quand vous cherchez un objet sur LeBonCoin, vous laissez plusieurs informations *implicites* sur votre perception des résultats proposés : les item sur lesquels vous avez cliqués bien sûr mais également ceux que vous avez probablement *vu* sans cliquer dessus ... Ces "vues sans clics" sont une précieuse information implicite sur les jugement que vous avez porté aux résultats proposés. Pour ce TP nous nous limiterons à ce concept de "vu x click" mais il est possible d'aller plus loin (dwell-time, hierarchisation des interactions explicites, ...). 

On appelle *Search Engine Results Page* - **SERP** - la liste des résultats classés par un SE. Un document qui figure dans les résulats d'une recherche a donc une position (son rang) au sein de la **SERP**.

Exemple, où :
- `query` est la recherche réalisée par un user et qui a débouché sur une SERP
- `clicked_id` : l'id de la bière cliquée par le user
- `user_id` l'id de l'utilisateur (simplifions en disant que c'est même l'id d'une recherche) : permet de retrouver tous les résultats proposés dans **une** recherche
- `id_in_serp` : l'id d'une bière figurant dans la SERP
- `pos_in_serp` : la position/le rang de la bière `id_in_serp` dans la SERP issue de la recherche 

In [ ]:
df_pref = spark.read.parquet("s3a://beer-dataset/beers_feedback")
df_pref.limit(3).show()

Un travail préliminaire au LTR est la constitution d'un dataset qui permet d'aggréger ces feedbacks laissés par tous les utilisateurs ayant réalisé la même query. Chacun a vu et cliqué selon ses propres impressions de pertinence et il convient de "moyenner" tout cela pour obtenir des appréciations globales. L'objectif d'un tel dataset est de pouvoir lister des exemples de triplets `(query, document, note)` qui permet de savoir que face à une *query* `milky stout low bitterness`, un *document* `Super bitter beer brewed with organic roasted barley and chocolate` aura une pertinence de *1/4* (arbitraire). 

Implémenter le modèle d'agrégation de feedback "cascade model" [1] (pour la culture, **inutile d'avoir lu l'article** pour le TD) qui propose une approche pragmatique pour obtenir ces données. La méthode est la suivante :
- pour chaque recherche utilisateur:
    - étudier la position de l'id cliqué dans la SERP - soit `clicked_pos_in_serp` cette information
    - Considérer que tout doc situés "au-dessus dans la SERP" (càd quand `pos_in_serp <= clicked_pos_in_serp`) avait été vu par l'utilisateur
    - Récapituler tous ces documents "vus et cliqués" et "vus mais pas cliqués"
- Pour chaque recherche et bière cliquée (`clicked_id`), calculer la "probabilité de clic sachant qu'elle a été vue", càd le nombre de fois qu'elle a été cliquée divisé par le nombre de fois où elle a été vue


[1] https://dl.acm.org/doi/abs/10.1145/1341531.1341545

# UC-3 : récupérer les docs qui parlent d'un mot

Peut-on utiliser SQL pour réaliser un mini moteur de recherche ? Pour différentes requêtes (`query` en anglais) textuelles très simples à base de mot-clef, retrouver les bières qui semblent répondre à la demande. Exemples :
- trouver les bières ou les brasseries qui parlent de bières "fine"
- idem pour "juicy"
- idem pour "genuine"
- idem pour les bières mâturées dans des "oak cask" (fûts en chêne) -> combien y en a-t-il ? $N_1$
   - idem pour les bières qui évoquent uniquement "cask" -> combien y en a-t-il ? $N_{1,1}$
   - idem pour celles ne parlant que de "oak" -> combien y en a-t-il ? $N_{1,2}$
- idem pour les bières qui évoquent "oak" et "cask" -> combien y en a-t-il ? $N_{2}$

In [ ]:
# your code

# UC-4 : vectorisation des description des bières
Préparer le recours à un service de vectorisation qui permettra de convertir la connaissance sur une bière en un vecteur numérique. Ce vecteur permet de sythétiser mathématiquement l'information disponible sur une bière et sa brasserie et pourra être réutilisé plus tard dans un moteur de recherche.
à faire :
- Préparer une description la plus complète possible pour chaque bière
- envoyer ces descriptions une à une via un appel HTTP sur le vectorizer (voir ci-dessous)

## Service de vectorisation 
Un faux service de vectorisation est disponible sur le server via [http://vectorizer:5000/vectorize] + un payload JSON ayant la clef `text` qui contient le texte à vectoriser.
Exemple de véritable service de vectorization: [https://jina.ai](https://jina.ai)

Essayons de construire d'avoir tous le même schéma de texte à vectoriser :
`the beer BEER_NAME from brewery BREWERY_NAME (BREWERY_DESCRIPTION) is defined as BEER_DESCRIPTION. Spec of the beer are: ABV=ABV_VALUE, IBU=IBU_VALUE, SRM=SRM_VALUE`

In [ ]:
# your code

# UC-5 : answer question in corpa

**Question difficile en Spark**

**Grandes lignes :** trouvons les documents qui répondent à une question. Exemple : à partir de la description vectorisée à UC-4 pour chaque bière, comment trouver les bières qui répondent à une description plus complète ? Exemple:
- "very bitter beer with smoky taste"
- "fruity sour - balanced sourness"
- "weird beer"

Voir la doc [Spark ML lib - feature extraction](https://spark.apache.org/docs/latest/api/python/reference/pyspark.mllib.html#feature) pour trouver des idées (TF-IDF, Word2Vec), ou utiliser le résultats de vos vectorisation de UC-4.

In [ ]:
queries = ["very bitter beer with smoky taste", "fruity sour - balanced sourness", "weird beer"]